In [11]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:  ", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:      ", torch.cuda.get_device_name(0))
    print("VRAM (GB):     ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("⚠️  No GPU detected — training will be very slow on CPU!")


CUDA available: True
Device count:   1
GPU name:       NVIDIA RTX 4000 Ada Generation
VRAM (GB):      21.5


In [12]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np

class CustomDataset(Dataset):
    """
    Custom dataset class for fine-tuning.
    Reads 'text_field' (text) and 'label' (numeric 0/1/2) columns from the CSV.
    Works with any AutoTokenizer-compatible model (roberta-base, xlm-roberta-base, etc.)
    """

    def __init__(self, csv_file, tokenizer, max_length=128):
        self.data = pd.read_csv(csv_file, low_memory=False)
        self.data = self.data.dropna(subset=['text_field', 'label'])
        self.data = self.data.reset_index(drop=True)

        self.data['label'] = self.data['label'].astype(int)
        self.unique_labels = sorted(self.data['label'].unique().tolist())

        self.tokenizer = tokenizer
        self.max_length = max_length

        print(f"Loaded {len(self.data)} examples")
        print(f"Unique labels: {self.unique_labels}")
        print(f"Label distribution:\n{self.data['label'].value_counts().sort_index()}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text  = str(self.data.loc[idx, 'text_field'])
        label = int(self.data.loc[idx, 'label'])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(label, dtype=torch.long)
        }

    def get_label_names(self):
        return [str(l) for l in self.unique_labels]


class CustomSubset(Dataset):
    """Wraps a CustomDataset with a list of integer indices."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices  = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.dataset[self.indices[idx]]


from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy':  float(accuracy_score(labels, predictions)),
        'f1':        float(f1_score(labels, predictions, average='weighted')),
        'precision': float(precision_score(labels, predictions, average='weighted')),
        'recall':    float(recall_score(labels, predictions, average='weighted')),
        'macro_f1':  float(f1_score(labels, predictions, average='macro')),
    }

print("✅ Classes and helpers defined.")


✅ Classes and helpers defined.


In [17]:
# ─── Configuration ─────────────────────────────────────────────────────────────
CSV_FILE_PATH  = r"C:\Users\22K-2127\Desktop\FYP\balanced_bias_news_dataset.csv"

MODEL_NAME     = "roberta-large"
MAX_LENGTH     = 512              # ↑ full context (was 256) — captures more text signal

BATCH_SIZE               = 4     # ↓ batch smaller so 512-token seqs fit VRAM
GRAD_ACCUM_STEPS         = 8     # effective batch = 32 (same eff. batch as best run)

EPOCHS                   = 10
LEARNING_RATE            = 1e-5   # ← proven best (the 80.29% run used this)
weight_decay             = 0.01
warmup_ratio             = 0.08   # 8% warmup — balanced between 6% (best) and 10%
lr_scheduler_type        = "cosine"
label_smoothing_factor   = 0.05
early_stopping_patience  = 3
load_best_model_at_end   = True
metric_for_best_model    = "eval_loss"

OUTPUT_DIR = "./roberta-large-finetuned-v3"

print(f"{'='*55}")
print(f"  MODEL      : {MODEL_NAME}")
print(f"  MAX_LENGTH : {MAX_LENGTH}  (↑ 256→512 for more context)")
print(f"  LR         : {LEARNING_RATE}  (proven best from 80.29% run)")
print(f"  Weight decay: {weight_decay}")
print(f"  Eff. batch : {BATCH_SIZE * GRAD_ACCUM_STEPS}  (batch={BATCH_SIZE} × accum={GRAD_ACCUM_STEPS})")
print(f"  Warmup     : {warmup_ratio*100:.0f}%")
print(f"  Epochs     : up to {EPOCHS}  with early stopping")
print(f"  Output     : {OUTPUT_DIR}")
print(f"{'='*55}")


  MODEL      : roberta-large
  MAX_LENGTH : 512  (↑ 256→512 for more context)
  LR         : 1e-05  (proven best from 80.29% run)
  Weight decay: 0.01
  Eff. batch : 32  (batch=4 × accum=8)
  Warmup     : 8%
  Epochs     : up to 10  with early stopping
  Output     : ./roberta-large-finetuned-v3


In [18]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create dataset
print("Creating dataset...")
dataset = CustomDataset(
    csv_file=CSV_FILE_PATH,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

# Split into train / validation (80/20 stratified)
train_data, val_data = train_test_split(
    dataset.data,
    test_size=0.2,
    random_state=42,
    stratify=dataset.data['label']
)

# Build subset wrappers
train_dataset = CustomSubset(dataset, train_data.index.tolist())
val_dataset   = CustomSubset(dataset, val_data.index.tolist())

print(f"Training examples:   {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")


Loading tokenizer...
Creating dataset...
Creating dataset...
Loaded 10020 examples
Unique labels: [0, 1, 2]
Label distribution:
label
0    3340
1    3340
2    3340
Name: count, dtype: int64
Training examples:   8016
Validation examples: 2004
Loaded 10020 examples
Unique labels: [0, 1, 2]
Label distribution:
label
0    3340
1    3340
2    3340
Name: count, dtype: int64
Training examples:   8016
Validation examples: 2004


In [19]:

from transformers import EarlyStoppingCallback
import torch

# ─── Load model ───────────────────────────────────────────────────────────────
print("Loading roberta-large...")
num_labels = len(dataset.unique_labels)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: str(l) for i, l in enumerate(dataset.unique_labels)},
    label2id={str(l): l for l in dataset.unique_labels},
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"Mixed precision: {'bf16' if use_bf16 else 'fp16'}")

# ─── Training arguments ───────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_ratio=warmup_ratio,
    weight_decay=weight_decay,
    max_grad_norm=1.0,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=False,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=lr_scheduler_type,
    label_smoothing_factor=label_smoothing_factor,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
    save_total_limit=1,
    save_only_model=True,
    dataloader_num_workers=0,
    seed=42,
)

# ─── Trainer (standard HF optimizer — proven best for roberta-large) ──────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
)

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * EPOCHS
print(f"\nTotal optimiser steps : {total_steps}")
print(f"Early stopping        : patience={early_stopping_patience} epochs on val_loss")
print(f"\n🚀 Starting training...")
trainer.train()

# ─── Save ─────────────────────────────────────────────────────────────────────
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Training complete!  Model saved to: {OUTPUT_DIR}")


Loading roberta-large...


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1233.35it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consi

Parameters: 355,362,819
Mixed precision: bf16

Total optimiser steps : 2500
Early stopping        : patience=3 epochs on val_loss

🚀 Starting training...

Total optimiser steps : 2500
Early stopping        : patience=3 epochs on val_loss

🚀 Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Macro F1
1,5.636332,0.625006,0.784930,0.784122,0.786915,0.784930,0.784122
2,4.215940,0.527022,0.836327,0.836486,0.840041,0.836327,0.836486
3,3.330862,0.549657,0.840818,0.840093,0.843496,0.840818,0.840093
4,2.482457,0.530965,0.854291,0.853891,0.854407,0.854291,0.853891


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]



SafetensorError: Error while serializing: I/O error: There is not enough space on the disk. (os error 112)

In [20]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import json, shutil, os
from datetime import datetime

# ─── Full evaluation ──────────────────────────────────────────────────────────
preds  = trainer.predict(val_dataset)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

label_names = ["Left (0)", "Center (1)", "Right (2)"]

print("=" * 60)
print("        CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(y_true, y_pred, target_names=label_names, digits=4, output_dict=True)
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

print("=" * 60)
print("        CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_true, y_pred)
print(f"{'':15s} {'Pred Left':>12} {'Pred Center':>12} {'Pred Right':>12}")
for i, row_label in enumerate(label_names):
    print(f"{row_label:15s} {cm[i,0]:>12} {cm[i,1]:>12} {cm[i,2]:>12}")

overall_acc = float(np.mean(y_true == y_pred))
print(f"\n✅ Overall Accuracy: {overall_acc*100:.2f}%")

# ─── Auto-save snapshot notebook + results log ────────────────────────────────
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
acc_str    = f"{overall_acc*100:.1f}"
snap_name  = f"fine_tuning__{MODEL_NAME.replace('/', '-')}__acc{acc_str}__{timestamp}.ipynb"
snap_path  = os.path.join(r"C:\Users\22K-2127\Desktop\FYP", snap_name)

# Copy the current notebook as a snapshot
src_nb = r"C:\Users\22K-2127\Desktop\FYP\fine_tuning.ipynb"
shutil.copy2(src_nb, snap_path)
print(f"\n📓 Snapshot saved → {snap_name}")

# ─── Append to results_log.csv ────────────────────────────────────────────────
log_path = r"C:\Users\22K-2127\Desktop\FYP\results_log.csv"
log_exists = os.path.exists(log_path)

run_record = {
    "timestamp":       timestamp,
    "notebook":        snap_name,
    "model":           MODEL_NAME,
    "max_length":      MAX_LENGTH,
    "batch_size":      BATCH_SIZE,
    "grad_accum":      GRAD_ACCUM_STEPS,
    "eff_batch":       BATCH_SIZE * GRAD_ACCUM_STEPS,
    "epochs_run":      int(trainer.state.epoch),
    "learning_rate":   LEARNING_RATE,
    "lr_scheduler":    lr_scheduler_type,
    "label_smoothing": label_smoothing_factor,
    "weight_decay":    weight_decay,
    "accuracy":        round(overall_acc * 100, 4),
    "f1_weighted":     round(report["weighted avg"]["f1-score"] * 100, 4),
    "f1_macro":        round(report["macro avg"]["f1-score"] * 100, 4),
    "precision":       round(report["weighted avg"]["precision"] * 100, 4),
    "recall":          round(report["weighted avg"]["recall"] * 100, 4),
    "output_dir":      OUTPUT_DIR,
}

import csv
with open(log_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=run_record.keys())
    if not log_exists:
        writer.writeheader()
    writer.writerow(run_record)

print(f"📊 Results logged  → results_log.csv")
print(f"\n{'='*60}")
print(f"  RUN SUMMARY")
print(f"{'='*60}")
print(f"  Model      : {MODEL_NAME}")
print(f"  Max length : {MAX_LENGTH}")
print(f"  Accuracy   : {overall_acc*100:.2f}%")
print(f"  F1 weighted: {report['weighted avg']['f1-score']*100:.2f}%")
print(f"  F1 macro   : {report['macro avg']['f1-score']*100:.2f}%")
print(f"{'='*60}")


        CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Left (0)     0.8511    0.9072    0.8783       668
  Center (1)     0.8585    0.8263    0.8421       668
   Right (2)     0.8536    0.8293    0.8413       668

    accuracy                         0.8543      2004
   macro avg     0.8544    0.8543    0.8539      2004
weighted avg     0.8544    0.8543    0.8539      2004

        CONFUSION MATRIX
                   Pred Left  Pred Center   Pred Right
Left (0)                 606           30           32
Center (1)                53          552           63
Right (2)                 53           61          554

✅ Overall Accuracy: 85.43%

📓 Snapshot saved → fine_tuning__roberta-large__acc85.4__20260304_130104.ipynb
📊 Results logged  → results_log.csv

  RUN SUMMARY
  Model      : roberta-large
  Max length : 512
  Accuracy   : 85.43%
  F1 weighted: 85.39%
  F1 macro   : 85.39%


In [22]:
import os

# ─── Save tokenizer directly into the best checkpoint (only ~5 MB) ────────────
BEST_CKPT = r"C:\Users\22K-2127\Desktop\FYP\roberta-large-finetuned-v3\checkpoint-1004"

# Save tokenizer files into the checkpoint folder
tokenizer.save_pretrained(BEST_CKPT)

# Verify
files = os.listdir(BEST_CKPT)
print("✅ checkpoint-1004/ contents:")
for f in sorted(files):
    size = os.path.getsize(os.path.join(BEST_CKPT, f))
    unit = "MB" if size > 1e6 else "KB"
    print(f"   {f:45s}  {size/1e6 if size>1e6 else size/1e3:6.1f} {unit}")

print(f"\n📦 Best model dir: {BEST_CKPT}")
print(f"   Accuracy: 85.43%  |  F1: 85.39%  |  roberta-large  |  MAX_LENGTH=512")


✅ checkpoint-1004/ contents:
   config.json                                       0.8 KB
   model.safetensors                               800.1 MB
   tokenizer.json                                    3.6 MB
   tokenizer_config.json                             0.4 KB

📦 Best model dir: C:\Users\22K-2127\Desktop\FYP\roberta-large-finetuned-v3\checkpoint-1004
   Accuracy: 85.43%  |  F1: 85.39%  |  roberta-large  |  MAX_LENGTH=512


In [1]:
import torch, os

# ── Save in-memory best model as pytorch_model.bin ────────────────────────────
# torch.load() uses map_location (no memory-mapping) → works with small pagefile
CKPT = r"C:\Users\22K-2127\Desktop\FYP\roberta-large-finetuned-v3\checkpoint-502"
OUT  = os.path.join(CKPT, "pytorch_model.bin")

print(f"Free disk: {os.statvfs(CKPT).f_bavail * os.statvfs(CKPT).f_frsize / 1e9:.1f} GB" if hasattr(os, "statvfs") else "")
print("Saving pytorch_model.bin from in-memory model weights...")
torch.save(model.state_dict(), OUT)
size = os.path.getsize(OUT) / 1e9
print(f"✅ Saved → {OUT}")
print(f"   Size : {size:.2f} GB")
print(f"\nThe web app will now load this .bin file instead of the safetensors file.")



Saving pytorch_model.bin from in-memory model weights...


NameError: name 'model' is not defined